In [23]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import date, datetime
import pandas as pd
from google.colab import drive

# Funções de extração de dados
def extrair_dados_produto(produto):
    try:
        # Extrair o título do produto
        nome = produto.find('div', class_='item-title').get_text(strip=True)

        # Extrair o link do produto
        link = produto.find('div', class_='item-title').a['href']

        # Extrair o preço do produto
        preco = produto.find('span', class_='price').get_text(strip=True)

        # Extrair a descrição do produto
        descricao = produto.find('div', class_='desc').get_text(strip=True)

        # Extrair a URL da imagem do produto
        imagem_url = produto.find('div', class_='thumb-wrapper').find('img')['src']

        # Extrair o SKU do produto, se disponível
        sku = produto.find('div', class_='desc').find('p').get_text(strip=True) if produto.find('div', class_='desc').find('p') else None

        # Salvar os dados em um dicionário
        dados_produto = {
            'titulo': nome,
            'link': link,
            'preco': preco,
            'descricao': descricao,
            'imagem_url': imagem_url,
            'sku': sku
        }

        return dados_produto
    except Exception as e:
        print(f"Erro ao extrair dados do produto: {e}")
        return None

def extrair_dados(url, category):
    try:
        print(f"Processando categoria '{category}' - URL: {url}")
        dados_produtos = []

        # Iteramos sobre todas as páginas
        page_num = 1
        while True:
            page_url = f"{url}/page/{page_num}/" if page_num > 1 else url
            print(f"Navegando na página {page_num}...")
            response = requests.get(page_url)
            soup = BeautifulSoup(response.text, 'html.parser')
            produtos = soup.find_all('li', class_='product-layout')

            if not produtos:
                break

            for produto in produtos:
                dados_produto = extrair_dados_produto(produto)
                if dados_produto:
                    dados_produtos.append(dados_produto)

            page_num += 1

        # Salvar os dados em CSV
        if dados_produtos:
            salvar_dados_csv(dados_produtos, category)

    except requests.RequestException as e:
        print(f"Erro ao fazer a requisição para a página {url}: {e}")
    except Exception as e:
        print(f"Erro ao extrair dados da página {url}: {e}")

# Função para salvar dados em CSV
def salvar_dados_csv(dados_produtos, category):
    try:
        df = pd.DataFrame(dados_produtos)

        # Montar o Google Drive
        drive.mount('/content/drive')

        # Definir o caminho de destino no Google Drive
        base_destination_dir = '/content/drive/MyDrive/WebScraping/FogoesShop/WebScraping_coleta_FogoesShop'
        today = datetime.today().strftime('%Y-%m-%d')
        destination_dir = os.path.join(base_destination_dir, today)

        # Verificar se o diretório já existe
        if not os.path.exists(destination_dir):
            os.makedirs(destination_dir, exist_ok=True)

        # Salvar o arquivo CSV no Google Drive
        file_path = os.path.join(destination_dir, f"{category}.csv")
        df.to_csv(file_path, index=False)
        print(f"Dados da categoria '{category}' salvos em: {file_path}")

        # Adicionar informações específicas
        print(f"ARQUIVO CRIADO {date.today()}")

    except Exception as e:
        print(f"Erro ao salvar dados em CSV: {e}")

# Função principal
def main():
    try:
        # Lista de links da Fogo E Shop com a categoria
        links = {
            "Acessórios para banheiros": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/"
            # Adicione mais links e categorias conforme necessário
        }

        # Iterar sobre as categorias e extrair dados
        for category, url in links.items():
            extrair_dados(url, category)

    except Exception as e:
        print(f"Erro no processo principal: {e}")

if __name__ == "__main__":
    main()


Processando categoria 'Acessórios para banheiros' - URL: https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/
Navegando na página 1...
Navegando na página 2...
Navegando na página 3...
Navegando na página 4...
Navegando na página 5...
Navegando na página 6...
Navegando na página 7...
Navegando na página 8...
Navegando na página 9...
Navegando na página 10...
Navegando na página 11...
Navegando na página 12...
Navegando na página 13...
Navegando na página 14...
Navegando na página 15...
Navegando na página 16...
Navegando na página 17...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dados da categoria 'Acessórios para banheiros' salvos em: /content/drive/MyDrive/WebScraping/FogoesShop/WebScraping_coleta_FogoesShop/2024-05-16/Acessórios para banheiros.csv
ARQUIVO CRIADO 2024-05-16
